# Model Routing for VSS (Switchyard)

Routes the VSS agent's LLM traffic, and optionally its VLM traffic, through a
model router
([NVIDIA-NeMo/Switchyard](https://github.com/NVIDIA-NeMo/Switchyard), Apache-2.0)
instead of a single fixed endpoint. VSS already supports remote
OpenAI-compatible LLM and VLM endpoints, so **no service is added to any VSS compose
file and no VSS profile is edited**.

<div style="color: #c00; border: 2px solid #c00; padding: 0.75em;">
<strong>NemoClaw order:</strong> Set <code>ROUTE_NEMOCLAW = "true"</code> in
section 1.3, run this Switchyard notebook through section 6, and leave the
router running. Then run <code>deploy_nemoclaw.ipynb</code>. This notebook
writes the router policy; section 3.2 of the NemoClaw notebook applies it.
</div>

This notebook:

1. builds Switchyard from a pinned source ref,
2. runs it as a local container and verifies real requests route across targets,
3. composes a VSS build that points `LLM_BASE_URL` (and, with `ROUTE_VLM`,
   RT-VLM's upstream) at the router, which prepares each VLM request's video
   for the model that answers it: frames for the judge and the capable VLM,
   the native clip for the efficient one.

The router serves a default policy, `switchyard/random`, which you can change
in 3.2.


## 1. Settings

### 1.1 Initialize upstream provider variables

Run this once, then set values in **one** option of 1.2.


In [ ]:
# ================== Upstream provider vars (set by ONE of (a)/(b) in 1.2) ==================
NVIDIA_API_KEY = UPSTREAM_BASE_URL = UPSTREAM_API_KEY = ""


### 1.2 Choose ONE upstream the router forwards to

The router holds no credential: callers present their own key and the
router forwards it upstream. VSS already sends its key as a bearer, so the
deployed path needs nothing extra.

#### (a) NVIDIA inference hub (default)


In [ ]:
# (a) NVIDIA inference hub. Prefer exporting NVIDIA_API_KEY in the environment
# before starting Jupyter; a value pasted here lands in the saved notebook.
UPSTREAM_BASE_URL = "https://inference-api.nvidia.com/v1"
NVIDIA_API_KEY = ""  # empty: read from the environment (preferred)


#### (b) Any OpenAI-compatible endpoint (self-hosted or other cloud)

In [ ]:
# (b) OpenAI-compatible endpoint. Prefer exporting UPSTREAM_API_KEY in the
# environment; a value pasted here lands in the saved notebook.
# UPSTREAM_BASE_URL = ""  # e.g. "http://my-endpoint:8000/v1"
# UPSTREAM_API_KEY  = ""  # empty: read from the environment (preferred)


### 1.3 Advanced settings (defaults; usually leave alone)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# ================== Default model-routing settings ==================
# Switchyard source pin. Always a commit SHA, never a branch name, so a routed
# evaluation can be repeated. PINNED TO AN OPEN PULL REQUEST: this is the head
# of Switchyard PR #790 (per-target media preparation and judge image limits),
# which no release carries yet. Move the pin to the squash commit on `main`
# when #790 merges, and to the v0.4.0 tag at release.
SWITCHYARD_REPO_URL = "https://github.com/NVIDIA-NeMo/Switchyard.git"
SWITCHYARD_REF = "82929f94a23825a9ab7d665dddb51c7589fa0301"
# Router runtime. The router is unauthenticated and forwards the caller's
# upstream credential. The default "local" publishes it on loopback plus
# the Docker bridge gateway: reachable from this host and its containers,
# unreachable from the outside network. Set this before the first run when
# the router must also serve NemoClaw. A true value rewrites ROUTER_BIND
# below to "0.0.0.0"; section 6 writes the handoff policy, and
# deploy_nemoclaw.ipynb applies it.
ROUTE_NEMOCLAW = "false"
ROUTER_PORT = "4000"
ROUTER_BIND = "local"
# "bridge" (default) publishes the container port onto loopback and the Docker
# bridge gateway. "host" runs the router in the host's network namespace, so
# router traffic to an upstream on THIS host stays on loopback. CI uses "host"
# because container-to-host routing policy differs across runners.
ROUTER_NETWORK = "bridge"
ROUTER_CONTAINER = "vss-model-router"
# Two-tier target set. VSS's LLM_NAME selects a *route id*, not a model.
ROUTER_TARGET_CAPABLE = "openai/openai/gpt-5.6-sol"
ROUTER_TARGET_EFFICIENT = "openai/openai/gpt-5.6-luna"
ROUTER_ROUTE = "switchyard/random"
# VLM routing, off by default. A judge sees each request's video and question,
# forecasts whether the efficient VLM would answer correctly, and the request
# goes to the efficient VLM when that forecast clears ROUTER_VLM_THRESHOLD,
# else to the capable one. The router prepares the media per target: up to 24
# frames for the capable VLM, the native clip for the efficient VLM, and for
# the judge the native clip when it is the efficient VLM itself (Switchyard
# prepares media once per model id within a route) or four frames when it is
# another model. VSS's VLM_NAME selects the route id.
ROUTE_VLM = "false"
# The judge is Cosmos 3 Super, the model family the routed figures were
# measured with. Any OpenAI-compatible VLM on the upstream works here,
# including a frontier model for a stronger judge at a higher cost.
ROUTER_VLM_JUDGE = "nvidia/nvidia/cosmos3-super-reasoner"
ROUTER_VLM_TARGET_CAPABLE = "openai/openai/gpt-6-astra"
ROUTER_VLM_TARGET_EFFICIENT = "nvidia/nvidia/cosmos3-super-reasoner"
ROUTER_VLM_ROUTE = "switchyard/vlm-judge"
# Lowest judge forecast (0 to 1) that keeps a request on the efficient VLM;
# lower sends it more traffic. An empty prompt uses the rubric in 3.2.
ROUTER_VLM_THRESHOLD = "0.48"
ROUTER_VLM_JUDGE_PROMPT = ""
# The router decodes video on the GPU (NVDEC). "auto" attaches the host GPU
# to the container when ROUTE_VLM is on; "none" never does (text-only CI).
ROUTER_GPU = "auto"
# Largest request body the router buffers, inline video included (MiB).
ROUTER_MAX_REQUEST_BODY_MB = "32"
# Which VSS Foundation the routed build is composed against.
VSS_FOUNDATION_PROFILE = "dev-profile-lvs"
# Both off by default: the notebook proves the path and deploys nothing.
MODEL_ROUTING_APPLY = "false"   # "true": print the exact deploy command (still not run)
ROUTER_TEARDOWN = "false"       # "true": remove the router container at the end (CI does)

# ================== Derived (no need to touch) ==================
HOME_DIR = Path.home().resolve()
VSS_REPO_DIR = Path(os.environ.get("VSS_REPO_DIR", Path.cwd())).resolve()
if not (VSS_REPO_DIR / "deploy" / "docker" / "compose.yml").is_file():
    # Notebook lives in deploy/docker/scripts/; walk up when run in place.
    for parent in Path.cwd().resolve().parents:
        if (parent / "deploy" / "docker" / "compose.yml").is_file():
            VSS_REPO_DIR = parent
            break
WORK_DIR = Path(os.environ.get("MODEL_ROUTING_WORK_DIR", HOME_DIR / "vss-model-routing")).resolve()
BUILD_DIR = VSS_REPO_DIR / "_builds" / "model-routing"
SWITCHYARD_DIR = WORK_DIR / "switchyard"
ROUTER_CONFIG = WORK_DIR / "config.toml"
ROUTER_IMAGE = f"switchyard:{SWITCHYARD_REF[:12]}"
ROUTER_PORT = int(str(ROUTER_PORT))
ROUTER_URL = f"http://127.0.0.1:{ROUTER_PORT}"

NVIDIA_API_KEY = (NVIDIA_API_KEY or os.environ.get("NVIDIA_API_KEY", "")).strip()
UPSTREAM_API_KEY = (UPSTREAM_API_KEY or os.environ.get("UPSTREAM_API_KEY", "")
                    or NVIDIA_API_KEY).strip()
_truthy = ("1", "true", "yes", "on")
ROUTE_NEMOCLAW = str(ROUTE_NEMOCLAW).strip().lower() in _truthy
ROUTE_VLM = str(ROUTE_VLM).strip().lower() in _truthy
ROUTER_VLM_THRESHOLD = float(str(ROUTER_VLM_THRESHOLD))
assert 0.0 <= ROUTER_VLM_THRESHOLD <= 1.0, "ROUTER_VLM_THRESHOLD must be between 0 and 1"
ROUTER_GPU = str(ROUTER_GPU).strip().lower()
assert ROUTER_GPU in ("auto", "none"), "ROUTER_GPU must be auto or none"
ROUTER_MAX_REQUEST_BODY_BYTES = int(str(ROUTER_MAX_REQUEST_BODY_MB)) * 1024 * 1024
if ROUTE_NEMOCLAW:
    # OpenShell may use a different Docker bridge than VSS. Opting in makes
    # the host endpoint reachable before NemoClaw onboards against it.
    ROUTER_BIND = "0.0.0.0"
MODEL_ROUTING_APPLY = str(MODEL_ROUTING_APPLY).strip().lower() in _truthy
ROUTER_TEARDOWN = str(ROUTER_TEARDOWN).strip().lower() in _truthy

FOUNDATION_ENV = (VSS_REPO_DIR / "deploy" / "docker" / "developer-profiles"
                  / VSS_FOUNDATION_PROFILE / "overrides.env")
ROOT_COMPOSE = VSS_REPO_DIR / "deploy" / "docker" / "compose.yml"
# The lvs profile ships /path/to placeholders for these two; real values are
# required for `docker compose config` to resolve.
os.environ["VSS_APPS_DIR"] = str(VSS_REPO_DIR / "deploy" / "docker")
os.environ.setdefault("VSS_DATA_DIR", str(WORK_DIR / "vss-apps-data"))

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"MODEL_ROUTING_UPSTREAM: {UPSTREAM_BASE_URL}")
print(f"MODEL_ROUTING_ROUTE: {ROUTER_ROUTE}")
print(f"VLM_ROUTING: {ROUTE_VLM}; route={ROUTER_VLM_ROUTE}; judge={ROUTER_VLM_JUDGE}; "
      f"threshold={ROUTER_VLM_THRESHOLD}; gpu={ROUTER_GPU}")
print(f"NEMOCLAW_ROUTING: {ROUTE_NEMOCLAW}; bind={ROUTER_BIND}")
print(f"Switchyard ref: {SWITCHYARD_REF}")
print(f"Work dir: {WORK_DIR}")
print(f"Build dir: {BUILD_DIR}")


## 2. Preflight

In [ ]:
import socket

RED, GREEN, YELLOW, RESET = "\033[31m", "\033[32m", "\033[33m", "\033[0m"


def port_is_free(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) != 0


def router_already_ours() -> bool:
    r = subprocess.run(["docker", "ps", "--filter", f"name=^{ROUTER_CONTAINER}$",
                        "--format", "{{.Names}}"], capture_output=True, text=True)
    return r.stdout.strip() == ROUTER_CONTAINER


required_checks = {
    "Upstream endpoint set": bool(UPSTREAM_BASE_URL),
    "Upstream API key set": bool(UPSTREAM_API_KEY),
    "SWITCHYARD_REF pinned": bool(SWITCHYARD_REF),
    "VSS repo layout (deploy/docker/compose.yml)": ROOT_COMPOSE.is_file(),
    f"Foundation profile ({VSS_FOUNDATION_PROFILE})": FOUNDATION_ENV.is_file(),
    "docker": shutil.which("docker") is not None,
    "docker compose": subprocess.run(["docker", "compose", "version"],
                                     capture_output=True).returncode == 0,
    "git": shutil.which("git") is not None,
    "curl": shutil.which("curl") is not None,
    f"port {ROUTER_PORT} free (or held by {ROUTER_CONTAINER})":
        port_is_free(ROUTER_PORT) or router_already_ours(),
}

failed = [name for name, ok in required_checks.items() if not ok]
for name, ok in required_checks.items():
    print(f"  {GREEN + 'ok ' if ok else RED + 'MISSING'}{RESET}  {name}")
if failed:
    raise RuntimeError(f"Preflight failed: {', '.join(failed)}")
print(f"{GREEN}Preflight passed.{RESET}")


## 3. Build and run the router

### 3.1 Fetch and build Switchyard (pinned)

Builds from the repository's own `Dockerfile`: a `debian:bookworm-slim`
runtime with the `switchyard-server` binary, CA certificates and libvpx,
running as a non-root user on port 4000. Video decoding runs on the GPU
through NVDEC, so with `ROUTE_VLM` the container starts with the host GPU
attached (3.3). The pin is a commit on an open Switchyard pull request (1.3);
the build fetches that commit by SHA. **No image is pinned or pulled from a
registry**; the pin is a source ref. Safe to re-run: an existing checkout is
reset to the pin and the Docker layer cache makes rebuilds cheap.


In [ ]:
def run(cmd, check=True, echo=True, **kw):
    print("$", " ".join(map(str, cmd)))
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True, **kw)
    if echo and r.stdout.strip():
        print(r.stdout.strip()[-2000:])
    if r.returncode != 0:
        print(r.stderr.strip()[-2000:])
        if check:
            raise RuntimeError(f"command failed ({r.returncode}): {cmd[0]}")
    return r


if not (SWITCHYARD_DIR / ".git").is_dir():
    run(["git", "clone", "--filter=blob:none", SWITCHYARD_REPO_URL, SWITCHYARD_DIR])
run(["git", "-C", SWITCHYARD_DIR, "fetch", "origin", SWITCHYARD_REF], echo=False)
# -f: a rerun against a dirty checkout must still land exactly on the pin.
run(["git", "-C", SWITCHYARD_DIR, "checkout", "-f", "--detach", SWITCHYARD_REF], echo=False)
head = run(["git", "-C", SWITCHYARD_DIR, "rev-parse", "HEAD"], echo=False).stdout.strip()
assert head == SWITCHYARD_REF, f"checkout drifted: {head}"
print(f"Switchyard at {head}")

run(["docker", "build", "-t", ROUTER_IMAGE, SWITCHYARD_DIR], echo=False)
print(f"Built {ROUTER_IMAGE}")


### 3.2 Write the router config

The schema belongs to Switchyard and moves with its releases. Two targets,
two routes: `switchyard/random` splits traffic ~50/50, `switchyard/control`
passes everything to the efficient target. Run the same work through both;
the difference is what routing bought.

`ROUTE_VLM` adds `switchyard/vlm-judge`, a capability route: the judge
(Cosmos 3 Super by default) reads the request's video and question and returns
a forecast of the efficient VLM's success. At or above `ROUTER_VLM_THRESHOLD` the efficient VLM answers; below
it, or when the verdict fails, the capable one does. Each answer model gets
its own copy of the media: the capable VLM up to 24 frames at 512 px, the
efficient VLM the native clip. The judge's rubric is the one the routed
figures were measured with, on video question answering;
`ROUTER_VLM_JUDGE_PROMPT` replaces it for other workloads. When the judge is the
efficient VLM itself, the default, that one target serves both roles and the
judge sees the native clip: Switchyard prepares media once per model id within
a route. A judge of another model runs on its own client entry and sees four
frames at 384 px, the protocol the routed figures were measured with.
`switchyard/vlm-control` passes everything to the efficient VLM.


In [ ]:
# VSS binds the Nemotron reasoning flag from the CONFIGURED model name
# (vss_agents/utils/reasoning_utils.py gates on "nemotron-3" being a substring).
# Under routing that name is a route id, so the gate never fires and a Nemotron
# target silently runs in its default reasoning mode: on one fixed aggregation
# prompt that was 7,358 tokens instead of 2,105. The router is the only
# component left that still knows which model serves the call, so it supplies
# the parameter per target. Switchyard merges extra_body into keys the caller
# omitted, so an explicit request value still wins.
def target_block(name, model_id):
    body = ""
    if "nemotron-3" in model_id.lower():   # same predicate reasoning_utils uses
        body = "\nextra_body = { chat_template_kwargs = { enable_thinking = true } }"
    return f'[targets.{name}]\nid = "{model_id}"\nllm_client = "upstream"{body}\n'


# The rubric the routed figures were measured with: a Cosmos 3 Super judge,
# four frames, Vantage video question answering. It names the packaged verdict
# fields Switchyard validates, so a replacement must keep them.
DEFAULT_VLM_JUDGE_PROMPT = """\
You are a probability forecaster for a model router that chooses which vision-language model answers one multiple-choice video question. You receive the question with its options and, when attached, a few evenly spaced frames of the video (sample times are labelled in seconds). Forecast one binary event: SUCCESS means the efficient model picks the correct option on one fresh run when given the whole video; FAILURE is any other outcome.

The efficient model is a small video-language model. Its capability card:
- SUP-1 [supported]: the answer is a scene-level attribute visible in any single frame (weather, lighting, road or room type, environment, colour or type of a large object, whether a kind of object or person is present).
- SUP-2 [supported]: the answer is the meaning of a static sign, marking or visible text that is large and clear.
- SUP-3 [supported]: the answer is one coarse, sustained action of the main subject that the frames show plainly (walking, sitting, carrying, driving straight).
- SUP-4 [supported]: the answer is a coarse spatial relation between two large static objects visible together in one frame.
- SUP-5 [supported]: the options differ so much that any frame rules all but one out.
- UNC-1 [uncertain]: the answer needs the exact count of static objects in one view, or a fine spatial relation between small objects.
- UNC-2 [uncertain]: the answer depends on what happens between the sampled frames but the options are coarse.
- LIM-1 [unsupported]: the answer requires counting events or repetitions over time, tracking entities entering or leaving, judging motion direction or trajectory, ordering events, or localising when something happened.
- LIM-2 [unsupported]: the answer requires fine-grained action distinctions, negation over the whole clip (which action is not present), small or brief details, or reading small text.

Procedure: state the crux in one sentence; pick the one rule that best fits the crux; set capability_boundary from the rule's bracket (SUP-* gives supported, UNC-* gives uncertain, LIM-* gives unsupported, none gives unmatched); then estimate p_solve as a natural frequency over 100 fresh runs. Use the frames, when present, as evidence of what is visible. Reserve 0.00 and 1.00 for impossible or certain outcomes. The routing threshold is not part of the forecast.

Return exactly one JSON object matching the response schema supplied with the request. No markdown, no commentary.
"""

vlm_judge_prompt = (ROUTER_VLM_JUDGE_PROMPT or DEFAULT_VLM_JUDGE_PROMPT).strip()
assert '"""' not in vlm_judge_prompt, "the judge prompt cannot contain a TOML multi-line delimiter"

# One media preparation per model id within a route: a judge that is the
# efficient VLM itself serves both roles and sees the native clip; another
# model gets its own client entry and the four-frame protocol.
same_model_judge = ROUTER_VLM_JUDGE == ROUTER_VLM_TARGET_EFFICIENT
judge_target = "vlm_efficient" if same_model_judge else "vlm_judge"
judge_blocks = "" if same_model_judge else f'''
# Same upstream as `upstream`. Switchyard keeps one media setting per model id
# and client, so the judge gets its own client entry.
[llm_clients.upstream_judge]
format = "openai_chat"
base_url = "{UPSTREAM_BASE_URL}"
forward_auth = true

[targets.vlm_judge]
id = "{ROUTER_VLM_JUDGE}"
llm_client = "upstream_judge"
# A deterministic verdict; judges that honor enable_thinking skip the reasoning block.
extra_body = {{ temperature = 0, chat_template_kwargs = {{ enable_thinking = false }} }}
[targets.vlm_judge.media]
video = "frames"
video_max_frames = 4
frame_max_edge = 384
image_max_edge = 384
max_input_bytes = 67108864
timeout_ms = 120000
'''

vlm_config = f'''{judge_blocks}
{target_block("vlm_capable", ROUTER_VLM_TARGET_CAPABLE)}[targets.vlm_capable.media]
video = "frames"
video_max_frames = 24
frame_max_edge = 512
max_input_bytes = 67108864
timeout_ms = 120000

{target_block("vlm_efficient", ROUTER_VLM_TARGET_EFFICIENT)}[targets.vlm_efficient.media]
video = "passthrough"

[routes.vlm_judge]
id = "switchyard/vlm-judge"
type = "llm_classifier"
mode = "capability"
classifier_target = "{judge_target}"
strong_target = "vlm_capable"
weak_target = "vlm_efficient"
base_threshold = {ROUTER_VLM_THRESHOLD}
threshold_step = 0.0
classify_trigger = "every_request"
max_output_tokens = 512
prompt = """
{vlm_judge_prompt}
"""

[routes.vlm_control]
id = "switchyard/vlm-control"
type = "passthrough"
target = "vlm_efficient"
''' if ROUTE_VLM else ""

ROUTER_CONFIG.write_text(f'''\
# Written by deploy_vss_switchyard.ipynb; verified against the pinned Switchyard commit (SWITCHYARD_REF).
schema_version = 1

[llm_clients.upstream]
format = "openai_chat"
base_url = "{UPSTREAM_BASE_URL}"
# The caller's own Authorization header is forwarded upstream: the router
# holds no credential, and an unauthenticated caller gets the upstream's 401.
forward_auth = true

{target_block("capable", ROUTER_TARGET_CAPABLE)}
{target_block("efficient", ROUTER_TARGET_EFFICIENT)}
[routes.random]
id = "switchyard/random"
type = "random"
targets = ["capable", "efficient"]

[routes.control]
id = "switchyard/control"
type = "passthrough"
target = "efficient"
{vlm_config}''')
print(f"Wrote {ROUTER_CONFIG}")


### 3.3 Start the router

The router writes a routing decision log to the work directory and forwards
the caller's Authorization header upstream. The default `local` mode publishes
only on loopback and the Docker bridge gateway. When `ROUTE_NEMOCLAW` is true,
it publishes on all host interfaces so OpenShell can reach it from another
bridge. Use that opt-in only on a trusted or firewalled host.

With `ROUTE_VLM` and `ROUTER_GPU = "auto"` the container gets the host GPU
for NVDEC: the nvidia runtime when Docker has one, else the CDI device, else
`--gpus`. The driver's video capability is requested explicitly, because
Docker's default capabilities leave the decode library out.
`ROUTER_MAX_REQUEST_BODY_MB` is the largest request the router buffers;
RT-VLM's inline chunks fit the default.


In [ ]:
import json as _json
import time
import urllib.request

gw = run(["docker", "network", "inspect", "bridge", "--format",
          "{{(index .IPAM.Config 0).Gateway}}"], check=False, echo=False).stdout.strip()
DOCKER_BRIDGE_GW = gw or "172.17.0.1"
def bridge_publish(bind):
    if bind == "local":
        return ["-p", f"127.0.0.1:{ROUTER_PORT}:4000",
                "-p", f"{DOCKER_BRIDGE_GW}:{ROUTER_PORT}:4000"]
    return ["-p", f"{bind}:{ROUTER_PORT}:4000"]

# CI uses host networking for its mock upstream, so keep both bridge-mode
# security choices checked here as part of every notebook run.
assert bridge_publish("local") == ["-p", f"127.0.0.1:{ROUTER_PORT}:4000",
                                    "-p", f"{DOCKER_BRIDGE_GW}:{ROUTER_PORT}:4000"]
assert bridge_publish("0.0.0.0") == ["-p", f"0.0.0.0:{ROUTER_PORT}:4000"]
if ROUTER_NETWORK == "host":
    # Host network namespace: no port publishing, the router listens on
    # ROUTER_PORT directly and reaches host-local upstreams over loopback.
    net_args = ["--network", "host"]
    publish = []
    router_listen_port = ROUTER_PORT
else:
    net_args = ["--add-host", "host.docker.internal:host-gateway"]
    publish = bridge_publish(ROUTER_BIND)
    router_listen_port = "4000"

def gpu_args():
    # NVDEC needs the driver's video capability, which Docker omits by default.
    caps = ["-e", "NVIDIA_DRIVER_CAPABILITIES=compute,utility,video"]
    runtimes = run(["docker", "info", "--format", "{{json .Runtimes}}"], check=False, echo=False).stdout
    if '"nvidia"' in runtimes:
        return ["--runtime", "nvidia", "-e", "NVIDIA_VISIBLE_DEVICES=all", *caps]
    if any(Path(p).is_file() for p in ("/etc/cdi/nvidia.yaml", "/var/run/cdi/nvidia.yaml")):
        return ["--device", "nvidia.com/gpu=all", *caps]
    return ["--gpus", "all", *caps]


gpu = gpu_args() if ROUTE_VLM and ROUTER_GPU == "auto" else []

run(["docker", "rm", "-f", ROUTER_CONTAINER], check=False, echo=False)
# The server runs as a non-root user, so the mounted log directory must be
# world-writable. The decision log is truncated so a rerun starts clean.
log_dir = WORK_DIR / "logs"
log_dir.mkdir(exist_ok=True)
log_dir.chmod(0o777)
(log_dir / "routing.jsonl").write_text("")
(log_dir / "routing.jsonl").chmod(0o666)

run(["docker", "run", "-d", "--name", ROUTER_CONTAINER,
     *net_args,
     *publish,
     *gpu,
     "-v", f"{ROUTER_CONFIG}:/etc/switchyard/config.toml:ro",
     "-v", f"{WORK_DIR / 'logs'}:/var/log/switchyard",
     ROUTER_IMAGE,
     "--config", "/etc/switchyard/config.toml", "--port", router_listen_port,
     "--routing-log-file", "/var/log/switchyard/routing.jsonl",
     "--max-request-body-bytes", str(ROUTER_MAX_REQUEST_BODY_BYTES)], echo=False)

deadline = time.time() + 60
while True:
    try:
        with urllib.request.urlopen(f"{ROUTER_URL}/v1/stats", timeout=5) as r:
            _json.loads(r.read())
        break
    except Exception:
        if time.time() > deadline:
            run(["docker", "logs", "--tail", "40", ROUTER_CONTAINER], check=False)
            raise RuntimeError("router did not become healthy within 60s")
        time.sleep(2)
print(f"Router healthy at {ROUTER_URL}" + (" with the GPU attached" if gpu else ""))


## 4. Verify routing end to end

Real requests through the router to the real upstream. Seeing both targets
served is the check that routing, not just proxying, happened; over 12
requests a single-target split has ~0.05% odds. With `ROUTE_VLM`, the same check then
runs on the VLM route, first with a still image and a question, then, when the
router has the GPU and `ffmpeg` is on this host, with a short generated clip,
the form RT-VLM sends by default: on `switchyard/vlm-judge` the judge must be
called once per request and a VLM target must answer each one; on
`switchyard/vlm-control` the efficient VLM answers them all.


In [ ]:
N_PROBES = 12
routing_log = WORK_DIR / "logs" / "routing.jsonl"


def log_records():
    return [_json.loads(line) for line in routing_log.read_text().splitlines()
            if line.strip()] if routing_log.is_file() else []


# Only records written from here on count, so the cell can be rerun on a live router.
seen = len(log_records())
payload = _json.dumps({
    "model": ROUTER_ROUTE,
    "messages": [{"role": "user", "content": "Reply with the single word: ok"}],
    "max_tokens": 8,
}).encode()

ok = 0
for _ in range(N_PROBES):
    req = urllib.request.Request(
        f"{ROUTER_URL}/v1/chat/completions", data=payload, method="POST",
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {UPSTREAM_API_KEY}"})
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            body = _json.loads(r.read())
        if body.get("choices"):
            ok += 1
    except Exception as exc:
        print(f"probe failed: {type(exc).__name__}: {exc}")

with urllib.request.urlopen(f"{ROUTER_URL}/v1/stats", timeout=10) as r:
    stats = _json.loads(r.read())
print(_json.dumps(stats, indent=2)[:1200])

# Decision-log schema at the pin: one line per upstream call, `model` = the
# model called; a judge call carries tier "classifier".
decisions = log_records()[seen:]
served = {}
for d in decisions:
    target = str(d.get("model") or "?")
    served[target] = served.get(target, 0) + 1

if ok < N_PROBES:
    raise RuntimeError(f"only {ok}/{N_PROBES} probes succeeded")
expected_targets = {ROUTER_TARGET_CAPABLE, ROUTER_TARGET_EFFICIENT}
if set(served) != expected_targets:
    raise RuntimeError(
        f"expected exactly the configured targets {sorted(expected_targets)} "
        f"served, got: {served or 'no decisions logged'}")
print(f"ROUTER_VERIFIED: {ok}/{N_PROBES} probes ok, served split {served}")

if ROUTE_VLM:
    import base64
    import struct
    import zlib

    # A 256 px red square, sent the way RT-VLM sends a frame: an inline image_url part.
    def png_chunk(kind, data):
        return (struct.pack(">I", len(data)) + kind + data
                + struct.pack(">I", zlib.crc32(kind + data) & 0xFFFFFFFF))
    side = 256
    png = (b"\x89PNG\r\n\x1a\n"
           + png_chunk(b"IHDR", struct.pack(">IIBBBBB", side, side, 8, 2, 0, 0, 0))
           + png_chunk(b"IDAT", zlib.compress(b"".join(b"\x00" + b"\xff\x00\x00" * side
                                                        for _ in range(side))))
           + png_chunk(b"IEND", b""))
    questions = ["Describe this image in one sentence.",
                 "Give the pixel coordinates of the top-left corner of the red area."]
    seen = len(log_records())
    vlm_ok = 0
    for i in range(N_PROBES):
        vlm_payload = _json.dumps({
            "model": ROUTER_VLM_ROUTE,
            "messages": [{"role": "user", "content": [
                {"type": "image_url", "image_url": {
                    "url": "data:image/png;base64," + base64.b64encode(png).decode(),
                    "detail": "auto"}},
                {"type": "text", "text": questions[i % 2]}]}],
            "max_tokens": 512,
        }).encode()
        req = urllib.request.Request(
            f"{ROUTER_URL}/v1/chat/completions", data=vlm_payload, method="POST",
            headers={"Content-Type": "application/json",
                     "Authorization": f"Bearer {UPSTREAM_API_KEY}"})
        try:
            with urllib.request.urlopen(req, timeout=300) as r:
                if _json.loads(r.read()).get("choices"):
                    vlm_ok += 1
        except Exception as exc:
            print(f"VLM probe failed: {type(exc).__name__}: {exc}")
    # Judge calls are logged with tier "classifier"; answers carry no tier.
    vlm_targets = {ROUTER_VLM_TARGET_CAPABLE, ROUTER_VLM_TARGET_EFFICIENT}
    judged, vlm_served = 0, {}
    for d in log_records()[seen:]:
        if d.get("tier") == "classifier" and d.get("model") == ROUTER_VLM_JUDGE:
            judged += 1
        elif d.get("tier") != "classifier" and d.get("model") in vlm_targets:
            vlm_served[d["model"]] = vlm_served.get(d["model"], 0) + 1
    if vlm_ok < N_PROBES:
        raise RuntimeError(f"only {vlm_ok}/{N_PROBES} VLM probes succeeded")
    if ROUTER_VLM_ROUTE == "switchyard/vlm-control":
        expected_vlm, expected_judged = {ROUTER_VLM_TARGET_EFFICIENT}, 0
    else:
        expected_vlm, expected_judged = vlm_targets, N_PROBES
    if judged != expected_judged:
        raise RuntimeError(f"expected {expected_judged} judge calls, got {judged}")
    if sum(vlm_served.values()) != N_PROBES or not set(vlm_served) <= expected_vlm:
        raise RuntimeError(f"expected {N_PROBES} answers from {sorted(expected_vlm)}, got: {vlm_served}")
    # The CI stub judge alternates its verdict, so its two targets must split the
    # probes evenly; a router that ignored the verdict, or always fell back, would
    # still pass the permissive real-upstream check above.
    if all(t.startswith("mock/") for t in vlm_targets) and ROUTER_VLM_ROUTE != "switchyard/vlm-control":
        if any(vlm_served.get(t) != N_PROBES // 2 for t in vlm_targets):
            raise RuntimeError(f"stub judge alternates verdicts, so each VLM target must serve {N_PROBES // 2}; got: {vlm_served}")
    print(f"VLM_ROUTER_VERIFIED: {vlm_ok}/{N_PROBES} image probes ok, "
          f"judge calls {judged}, served split {vlm_served}")

    # Native video, the form RT-VLM sends by default. Decoding needs the GPU in
    # the container, so this runs only when it was attached and ffmpeg can make a clip.
    if gpu and shutil.which("ffmpeg"):
        clip = WORK_DIR / "probe.mp4"
        run(["ffmpeg", "-y", "-v", "error", "-f", "lavfi", "-i", "testsrc=duration=2:size=320x240:rate=8",
             "-c:v", "libx264", "-pix_fmt", "yuv420p", clip], echo=False)
        seen = len(log_records())
        video_payload = _json.dumps({
            "model": ROUTER_VLM_ROUTE,
            "messages": [{"role": "user", "content": [
                {"type": "video_url", "video_url": {
                    "url": "data:video/mp4;base64," + base64.b64encode(clip.read_bytes()).decode()}},
                {"type": "text", "text": "Describe this video in one sentence."}]}],
            "max_tokens": 128,
        }).encode()
        req = urllib.request.Request(
            f"{ROUTER_URL}/v1/chat/completions", data=video_payload, method="POST",
            headers={"Content-Type": "application/json",
                     "Authorization": f"Bearer {UPSTREAM_API_KEY}"})
        with urllib.request.urlopen(req, timeout=300) as r:
            if not _json.loads(r.read()).get("choices"):
                raise RuntimeError("the video probe returned no choices")
        video_decisions = log_records()[seen:]
        judged_video = sum(1 for d in video_decisions if d.get("tier") == "classifier")
        answered = [d["model"] for d in video_decisions
                    if d.get("tier") != "classifier" and d.get("model") in vlm_targets]
        if ROUTER_VLM_ROUTE != "switchyard/vlm-control" and judged_video != 1:
            raise RuntimeError(f"expected one judge call for the video probe, got {judged_video}")
        if len(answered) != 1:
            raise RuntimeError(f"expected one VLM answer for the video probe, got {answered}")
        print(f"VLM_VIDEO_PROBE: clip decoded on the router; judge calls {judged_video}, answered by {answered[0]}")
    else:
        print("VLM_VIDEO_PROBE: skipped (no GPU attached to the router, or ffmpeg is not on this host)")


## 5. Point VSS at the router

This is a configuration of an endpoint VSS already supports:

| Variable | Value | Why |
|---|---|---|
| `LLM_MODE` | `remote` | existing supported override set |
| `LLM_NAME_SLUG` | `none` | with remote mode, resolves the local LLM NIM out of `COMPOSE_PROFILES`, freeing its GPU |
| `LLM_BASE_URL` | router address, **no trailing `/v1`** | the agent appends `/v1` itself |
| `LLM_NAME` | a route id (`switchyard/random`) | with a router, this selects a routing policy, not a model |
| `VLM_MODE`, `VLM_NAME_SLUG`, `VLM_PORT` | `remote`, `none`, `30082` | with `ROUTE_VLM`: the remote-VLM settings `dev-profile.sh --use-remote-vlm` writes |
| `RTVI_VLM_MODEL_TO_USE`, `RTVI_VLM_MODEL_PATH` | `openai-compat`, `none` | RT-VLM forwards every VLM call, the agent's and LVS's, instead of loading a model |
| `VLM_BASE_URL`, `RTVI_VLM_ENDPOINT` | router address, the same with `/v1` | RT-VLM's upstream is the router |
| `VLM_NAME` | a VLM route id (`switchyard/vlm-judge`) | RT-VLM serves it as its model id and forwards it as the route |

RT-VLM keeps its default `REMOTE_VIDEO_INPUT=true` and sends each chunk as one
H.264 clip; the router prepares that clip for whichever model answers (3.2).

Values are rewritten **in place**: Compose resolves dotenv references against
entries parsed *so far*, so moving a key after its consumers silently breaks
paths like `VST_CONFIG_PATH`. The checked-in profile is never edited.


In [ ]:
import re

BUILD_DIR.mkdir(parents=True, exist_ok=True)
override_env = BUILD_DIR / "override.env"


def host_routable_ip() -> str:
    # The host address a bridge-networked container can reach. A short
    # hostname is not resolvable from inside the containers; the primary
    # outbound IP is. No packet is sent by this probe.
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        s.connect(("8.8.8.8", 80))
        return s.getsockname()[0]


# The URL must match where the router listens. In "local" mode that is
# the Docker bridge gateway, which the VSS containers can reach.
if os.environ.get("MODEL_ROUTING_VSS_URL"):
    router_for_vss = os.environ["MODEL_ROUTING_VSS_URL"]
elif ROUTER_BIND == "local":
    router_for_vss = f"http://{DOCKER_BRIDGE_GW}:{ROUTER_PORT}"
elif ROUTER_BIND == "0.0.0.0":
    router_for_vss = f"http://{host_routable_ip()}:{ROUTER_PORT}"
else:
    router_for_vss = f"http://{ROUTER_BIND}:{ROUTER_PORT}"

text = FOUNDATION_ENV.read_text()
repoint = [("LLM_MODE", "remote"), ("LLM_NAME_SLUG", "none"),
           ("LLM_BASE_URL", router_for_vss), ("LLM_NAME", ROUTER_ROUTE)]
if ROUTE_VLM:
    repoint += [("VLM_MODE", "remote"), ("VLM_NAME_SLUG", "none"), ("VLM_PORT", "30082"),
                ("RTVI_VLM_MODEL_TO_USE", "openai-compat"), ("RTVI_VLM_MODEL_PATH", "none"),
                ("VLM_BASE_URL", router_for_vss), ("RTVI_VLM_ENDPOINT", f"{router_for_vss}/v1"),
                ("VLM_NAME", ROUTER_VLM_ROUTE)]
for key, value in repoint:
    text, n = re.subn(rf"^{key}=.*$", f"{key}={value}", text, flags=re.MULTILINE)
    if n != 1:
        raise RuntimeError(f"expected exactly one {key}= line in {FOUNDATION_ENV}, found {n}")
override_env.write_text(text)

# Validate against the checked-in root compose; its include paths are
# relative to its own directory, so it cannot be copied elsewhere.
# Compose interpolates process-environment secrets into its output, so every
# config call runs with the secrets scrubbed: resolved.yml is a deployable
# artifact, not a credential store.
scrubbed_env = {**os.environ}
for secret in ("NVIDIA_API_KEY", "UPSTREAM_API_KEY", "NGC_CLI_API_KEY",
               "OPENAI_API_KEY", "ANTHROPIC_API_KEY", "RTVI_VLM_API_KEY"):
    scrubbed_env.pop(secret, None)
run(["docker", "compose", "--env-file", override_env,
     "-f", ROOT_COMPOSE, "config", "--quiet"], echo=False, env=scrubbed_env)


def service_set(env_file):
    r = run(["docker", "compose", "--env-file", env_file, "-f", ROOT_COMPOSE,
             "config", "--services"], echo=False, env=scrubbed_env)
    return set(r.stdout.split())


foundation, routed = service_set(FOUNDATION_ENV), service_set(override_env)
added, removed = sorted(routed - foundation), sorted(foundation - routed)
if added:
    raise RuntimeError(f"routing must add no service, added: {added}")
print(f"Services: {len(foundation)} -> {len(routed)}; added none; "
      f"resolved away: {removed}")

resolved = run(["docker", "compose", "--env-file", override_env,
                "-f", ROOT_COMPOSE, "config"], echo=False, env=scrubbed_env)
for secret_value in (NVIDIA_API_KEY, UPSTREAM_API_KEY):
    if secret_value and secret_value in resolved.stdout:
        raise RuntimeError("a credential reached the resolved compose output")
(BUILD_DIR / "resolved.yml").write_text(resolved.stdout)
print(f"VSS_ROUTING_COMPOSE: valid ({BUILD_DIR})")

if MODEL_ROUTING_APPLY:
    print("\nTo deploy the routed VSS build:")
    print(f"  docker compose --env-file {override_env} -f {ROOT_COMPOSE} up -d")
    # The key checked above is the one THIS notebook used to verify the router.
    # VSS reads its own: the agent takes NVIDIA_API_KEY / OPENAI_API_KEY, and the
    # summarizer takes NVIDIA_API_KEY (services/video-summarization/configs/
    # config.yaml). Because the router forwards the caller's header, that key has
    # to be the real upstream credential, so export it before the command above.
    print("  Export NVIDIA_API_KEY (and OPENAI_API_KEY if your profile uses it)")
    print("  first: the router forwards whatever VSS sends, and VSS does not read")
    print("  UPSTREAM_API_KEY.")
    if ROUTE_VLM:
        print("  With ROUTE_VLM, also export RTVI_VLM_API_KEY: RT-VLM forwards that key")
        print("  (else NGC_CLI_API_KEY), not NVIDIA_API_KEY.")
else:
    print(f"\nRouted build ready in {BUILD_DIR}; set MODEL_ROUTING_APPLY=\"true\" for the deploy command.")


## 6. Route the NemoClaw agent (optional)

Section 5 repoints the VSS deployment, whose `vss-agent` service reads
`LLM_BASE_URL`. The NemoClaw sandbox agent is configured separately, through
its own provider settings, and a router is just an OpenAI-compatible endpoint
to it. This cell prints the values to paste into the option (a) cell of
[`deploy_nemoclaw.ipynb`](deploy_nemoclaw.ipynb) section 1.2, the
OpenAI-compatible endpoint option.

Two things the sandbox needs:

- **Reachability.** The sandbox reaches host services as
  `host.openshell.internal`, which maps to a Docker bridge gateway. A router
  published only on loopback is not reachable from it.
- **Egress policy.** The sandbox allowlists host ports individually, and the
  shipped policy does not open the router's port: an operator who is not
  routing the agent should not have the port opened. The cell below
  writes a separate preset that opens it only for this deployment; section 3.2
  of `deploy_nemoclaw.ipynb` applies it automatically when present.

Start the router before section 3.1 of `deploy_nemoclaw.ipynb`, so the endpoint
is live when NemoClaw onboards against it.


In [ ]:
nemoclaw_port = ROUTER_PORT
nemoclaw_url = f"http://host.openshell.internal:{nemoclaw_port}/v1"

# OpenShell may sit on a different bridge than VSS. ROUTE_NEMOCLAW selects
# the broader bind before section 3.3 starts the router.
reachable = ROUTER_BIND == "0.0.0.0"
print('NEMOCLAW_PROVIDER     = "custom"')
print(f"NEMOCLAW_ENDPOINT_URL = {nemoclaw_url!r}")
print(f"NEMOCLAW_MODEL        = {ROUTER_ROUTE!r}")
print("COMPATIBLE_API_KEY    = <your upstream key; the router forwards it>")
print()
if reachable:
    print("Router publishes on all interfaces, so the sandbox can reach it.")
else:
    print(f"NOTE: the router is published on {ROUTER_BIND!r}. For NemoClaw, set")
    print('      ROUTE_NEMOCLAW = "true" in 1.3 and rerun from there, otherwise')
    print("      the sandbox cannot reach it.")

# A separate policy, never one the shipped file defines. Policies merge into
# the live sandbox policy BY NAME and override on collision, so reusing
# vss-backend-readwrite here would replace it instead of adding to it.
policy_fragment = WORK_DIR / "vss_model_router_policy.yaml"
policy_fragment.write_text("\n".join([
    "# Generated by deploy_vss_switchyard.ipynb. Opens the sandbox's egress to a",
    "# model router running on this host. Apply only when routing the agent.",
    "# The allowed_ips below are the ranges Docker may have handed OpenShell:",
    "# this notebook can run before the sandbox exists, so it cannot detect them.",
    "# Section 3.2 of deploy_nemoclaw.ipynb narrows them to the sandbox's own.",
    "preset:",
    "  name: vss-model-router",
    '  description: "Sandbox egress to a model router on this host"',
    "",
    "network_policies:",
    "  vss-model-router:",
    "    name: vss-model-router",
    "    endpoints:",
    "      - host: host.openshell.internal",
    f"        port: {nemoclaw_port}",
    "        access: full",
    "        allowed_ips:",
    "          - 172.17.0.0/16",
    "          - 172.18.0.0/16",
    "          - 172.20.0.0/16",
    "    # OpenShell enforces on the kernel-trusted /proc/<pid>/exe, so the",
    "    # binaries that make the calls have to be listed or the port stays",
    "    # denied. Same set the VSS backend policy uses.",
    "    binaries:",
    "      - { path: /usr/bin/curl }",
    "      - { path: /usr/local/bin/node }",
    "      - { path: /usr/local/bin/openclaw }",
    "      - { path: /usr/bin/python3 }",
    "      - { path: /usr/bin/python3.13 }",
    "",
]))
revoke_fragment = WORK_DIR / "vss_model_router_policy_revoke.yaml"
revoke_fragment.write_text("\n".join([
    "# Generated by deploy_vss_switchyard.ipynb. Closes the router port again.",
    "# Policies merge by name and override on collision, so applying this",
    "# replaces the vss-model-router policy with one that allows nothing.",
    "preset:",
    "  name: vss-model-router",
    '  description: "Router egress revoked"',
    "",
    "network_policies:",
    "  vss-model-router:",
    "    name: vss-model-router",
    "    endpoints: []",
    "    binaries: []",
    "",
]))
# Revocation works because policy-add OVERRIDES BY NAME: re-adding a preset
# whose name matches an applied one replaces its recorded entries (NemoClaw
# policy-channel.ts at the pinned CLI tag: 'A re-add replaces the recorded
# entries'). `policy remove vss-model-router` is the other route. That makes the shared name the
# load-bearing invariant, so verify it mechanically: if a future edit lets
# the names drift, or the revoke fragment stops being empty, revocation
# silently becomes a no-op. CI requires the marker line below.
_names = lambda text: sorted({
    line.split("name:", 1)[1].strip()
    for line in text.splitlines()
    if line.strip().startswith("name:")})
_grant, _revoke = policy_fragment.read_text(), revoke_fragment.read_text()
assert _names(_grant) == _names(_revoke) == ["vss-model-router"], \
    "grant and revoke must target the same preset/policy name"
assert "endpoints: []" in _revoke and "binaries: []" in _revoke, \
    "the revoke fragment must allow nothing"
print("ROUTER_POLICY_LIFECYCLE: grant/revoke share name vss-model-router; revoke allows nothing")

print(f"Wrote {policy_fragment}")
print("Section 3.2 of deploy_nemoclaw.ipynb applies this automatically when it")
print("sits at the default path, including after a re-onboard. Point that")
print("notebook elsewhere with MODEL_ROUTER_POLICY if you moved the work dir.")
print("Nothing opens the port until that cell runs: on a sandbox that is already")
print("onboarded, re-run 3.2 now. It needs no re-onboard, and it narrows the")
print("allowed_ips above to the networks the sandbox is really on.")
print()
if reachable:
    print(f"NEMOCLAW_ROUTING_READY: {nemoclaw_url}")
else:
    print("NEMOCLAW_ROUTING_SKIPPED: router is not bound for NemoClaw")


## 7. Teardown (optional)

Off by default so an interactive session can keep using the router. CI sets
`ROUTER_TEARDOWN=true`.


In [ ]:
if ROUTER_TEARDOWN:
    # check=True: "done" must mean the container is actually gone.
    run(["docker", "rm", "-f", ROUTER_CONTAINER], echo=False)
    print("ROUTER_TEARDOWN: done")
else:
    print(f"ROUTER_TEARDOWN: skipped; router still serving at {ROUTER_URL}")
revoke = WORK_DIR / "vss_model_router_policy_revoke.yaml"
grant = revoke.with_name("vss_model_router_policy.yaml")
manual_steps = (
    "To close the router port again, delete the preset so 3.2 stops applying\n"
    f"it (rm '{grant}'), then apply the revoke file once:\n"
    f"  <nemoclaw|nemohermes> <sandbox> policy-add --from-file '{revoke}' --yes\n"
    "  (policy-add OVERRIDES BY NAME: applying a preset whose name matches an\n"
    "   applied one replaces its contents, so this empty fragment closes the\n"
    "   port. Re-onboarding alone does not\n"
    "   revoke: deploy_nemoclaw.ipynb 3.2 reapplies the grant file on every\n"
    "   onboard for as long as it exists, which is why deleting it comes first.)"
)
# Revoke exactly what was granted: deploy_nemoclaw.ipynb 3.2 leaves one marker
# file per CLI and sandbox it applied the grant to, so no name is guessed here
# and a sandbox this grant never reached is never rewritten. A marker goes
# only after its own revoke has landed and only if 3.2 has not touched it since
# the revoke started, so a grant taken while this runs keeps its marker for the
# next teardown; the grant file goes only when no marker is left, and 3.2
# touches the marker before it adds the policy, so a grant in flight holds it.
# The revoke follows ROUTER_TEARDOWN on purpose: the grant exists to reach this
# router, so it goes when the router goes and stays while the router stays for
# the step that follows; CI's cleanup, which removes the router outside this
# notebook, revokes the same markers before it deletes the work directory.
markers = sorted(grant.parent.glob(f"{grant.stem}.applied.*.*"))
revoke_failed = []
if ROUTER_TEARDOWN and revoke.is_file() and markers:
    for marker in markers:
        cli, sandbox = marker.name.rsplit(".", 2)[1:]
        if not (shutil.which(cli) and shutil.which("openshell")):
            # Nothing was tried, so this is not a failed revoke; the steps below
            # tell the operator how to close the port from a shell that has the CLI.
            print(f"ROUTER_REVOKE: {cli} or openshell is not on PATH, so {sandbox} cannot be revoked from here; {marker.name} kept")
            continue
        touched = marker.stat().st_mtime_ns
        if run(["openshell", "sandbox", "get", sandbox], check=False, echo=False).returncode != 0:
            # A sandbox the gateway no longer lists took its grant with it (the
            # documented teardown destroys the sandbox before Compose comes down),
            # so its marker goes, unless 3.2 touched it again meanwhile. A gateway
            # that cannot be asked, or one that still lists the name, keeps the marker.
            listing = run(["openshell", "sandbox", "list"], check=False, echo=False)
            try:
                untouched = marker.stat().st_mtime_ns == touched
            except FileNotFoundError:
                untouched = False
            if listing.returncode == 0 and sandbox not in listing.stdout and untouched:
                marker.unlink()
                print(f"ROUTER_REVOKE: {sandbox} no longer exists, its grant went with it; {marker.name} removed")
            else:
                print(f"ROUTER_REVOKE: {sandbox} could not be revoked now; {marker.name} kept for the next teardown")
                revoke_failed.append(sandbox)
            continue
        if run([cli, sandbox, "policy-add", "--from-file", revoke, "--yes"], check=False).returncode != 0:
            print(f"ROUTER_REVOKE: failed on {sandbox}; {marker.name} kept for the next teardown")
            revoke_failed.append(sandbox)
            continue
        try:
            untouched = marker.stat().st_mtime_ns == touched
        except FileNotFoundError:
            untouched = False
        if untouched:
            marker.unlink()
            print(f"ROUTER_REVOKE: done on {sandbox}")
        elif not marker.exists():
            print(f"ROUTER_REVOKE: done on {sandbox}; its marker was already gone")
        else:
            print(f"ROUTER_REVOKE: {sandbox} was granted again meanwhile; {marker.name} kept for the next teardown")
    if not any(grant.parent.glob(f"{grant.stem}.applied.*.*")) and grant.is_file():
        grant.unlink()
        print(f"ROUTER_REVOKE: {grant.name} removed so 3.2 stops reapplying it")
    else:
        print(manual_steps)
    if revoke_failed:
        raise RuntimeError(f"router revoke failed on: {', '.join(revoke_failed)}; run this cell again")
elif revoke.is_file():
    print(manual_steps)
